In [ ]:
import os
import torch
import librosa
import numpy as np
import soundfile as sf
from scipy.io import wavfile
from IPython.display import Audio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Tokenizer

In [ ]:
def extract_text(audioFilePath, model, tokenizer):
    
    print(audioFilePath)
    
    data = wavfile.read(audioFilePath)
    framerate = data[0]
    sounddata = data[1]
    time = np.arange(0,len(sounddata))/framerate
    inputAudio, _ = librosa.load(audioFilePath, sr=16000)
    inputValues = tokenizer(inputAudio, return_tensors="pt").input_values
    logits = model(inputValues).logits
    predictedIds = torch.argmax(logits, dim=-1)
    transcription = tokenizer.batch_decode(predictedIds)[0]
    print(transcription)
    textName = audioFilePath.split(".")[0]
    with open(textName + '_text.txt', 'w') as f:     # 打开test.txt   如果文件不存在，创建该文件。
        f.write(transcription)  # 把变量var写入test.txt。这里var必须是str格式，如果不是，则可以转一下。

In [ ]:
# 预训练模型
WAV2VEC_MODEL_DIR = "facebook/wav2vec2-base-960h"
tokenizer = Wav2Vec2Tokenizer.from_pretrained(WAV2VEC_MODEL_DIR)
model = Wav2Vec2ForCTC.from_pretrained(WAV2VEC_MODEL_DIR)

# 数据集地址
DATASET_DIR = os.environ.get("FAKEAVCELEB_DIR", "data/FakeAVCeleb")
listType = os.listdir(DATASET_DIR)  # 列出文件夹下所有的目录与文件

In [ ]:
for i in range(0, len(listType)):
    if listType != 'RealVideo-RealAudio':
        pathType = os.path.join(DATASET_DIR, listType[i])
        # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\
        listLocation = os.listdir(pathType)
        for j in range(0, len(listLocation)):
            pathLocation = os.path.join(pathType, listLocation[j])
            # e.g:  ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\
            listGender = os.listdir(pathLocation)
            for k in range(0, len(listGender)):
                pathGender = os.path.join(pathLocation, listGender[k])
                # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\men\
                listId = os.listdir(pathGender)
                for l in range(0, len(listId)):
                    pathId = os.path.join(pathGender, listId[l])
                    # e.g: ..\FakeAVCeleb_v1.2\FakeVideo-FakeAudio\African\men\id001166
                    listVideo = os.listdir(pathId)
                    for file in listVideo:
                        pathFile = os.path.join(pathId, file)
                        
                        # 判断文件类型
                        ext = os.path.splitext(pathFile)[1]
                        if ext ==".wav":
                            extract_text(pathFile, model, tokenizer)